In [ ]:
drop_cols_df3 = [
    'browser_is_whale', 'browser_is_chrome', 'browser_is_edge',
    'email', 'name', 'browser_version', 'insert_id',
    'screen_height', 'screen_width', 'search_engine', 'lib_version',
    'mp_api_endpoint', 'mp_lib', 'mp_loader', 'extension_version',
    'llm_provider', 'llm_version', 'platform','device', 'identity_failure_reason', 'mp_sent_by_lib_version', 'page_url'
]

In [ ]:
drop_cols_df2 = [
    'browser_version', 'email', 'geo_source', 'name',
    'initial_utm_campaign', 'initial_utm_content', 'initial_utm_id',
    'initial_utm_medium', 'initial_utm_source', 'initial_utm_term',
    'initial_utm_source_platform'
]

In [ ]:
import matplotlib
import shutil
import os

# 1. 캐시 디렉토리 경로 확인
cache_dir = matplotlib.get_cachedir()

# 2. 캐시 디렉토리 삭제
if os.path.exists(cache_dir):
    shutil.rmtree(cache_dir)
    print(f"✅ 폰트 캐시가 삭제되었습니다: {cache_dir}")
else:
    print("이미 캐시가 삭제되었거나 존재하지 않습니다.")

# 3. 중요: 이 코드를 실행한 후에는 반드시 상단 메뉴의 [Kernel] -> [Restart Kernel]을 눌러주세요.

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import platform
import seaborn as sns
sns.set_theme(style="darkgrid")
import koreanize_matplotlib

In [ ]:
df = pd.read_csv('./data/full_merged_v3.csv')

In [ ]:
df.info()

In [ ]:
df.shape

### 사용 위치(position)별 사용빈도 분석

In [ ]:
df['position'].value_counts()

In [ ]:
df['platform'].value_counts()

In [ ]:
df[df['platform'].notnull()]['position'].value_counts()  

In [ ]:
# 1. 데이터 생성 (English columns)
data = {
    'Position': ['sidePanel', 'modal', 'popup'],
    'Count': [34114, 14807, 949]
}
df_table = pd.DataFrame(data)

# 2. Position 컬럼을 인덱스로 설정
df_table.set_index('Position', inplace=True)

# 3. 비율 계산 (Percentage)
total_count = df_table['Count'].sum()
df_table['Percentage (%)'] = (df_table['Count'] / total_count) * 100

# 4. 맨 아래에 'Total' 행 추가
df_table.loc['Total'] = [total_count, 100.0]

# 5. 결과 출력
print(" [!] Table generation complete")
display(df_table)

In [ ]:
# 1. 데이터 집계 및 정렬
# sidePanel(가장 큰 값)이 오른쪽으로 가도록 오름차순 정렬
df_counts = df['position'].value_counts().sort_values(ascending=True)

# 2. 색상 설정
color_map = {
    'sidePanel': 'mediumseagreen',
    'modal': 'steelblue',
    'popup': 'orange'
}
palette_colors = [color_map[label] for label in df_counts.index]

# 3. 그래프 그리기
print("  - [시각화] Position별 사용 빈도 막대 그래프 생성 중...")
plt.figure(figsize=(7, 5))

# [수정] hue에 x축과 동일한 값을 넣고, legend=False로 설정하여 경고 해결
ax = sns.barplot(
    x=df_counts.index,
    y=df_counts.values,
    hue=df_counts.index,  # x와 동일하게 설정
    palette=palette_colors,
    legend=False          # 범례 숨김 (hue 사용 시 자동 생성되는 것 방지)
)

# 제목 및 라벨 설정
plt.title('사용 위치(Position)별 사용 빈도', fontsize=16)
plt.ylabel('빈도 (건)', fontsize=12)
plt.xlabel('사용 위치', fontsize=12)

# 막대 위에 수치 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

# 저장 및 출력
plt.savefig('./output/images/position_count_bar_sorted.png')
print("  - 'position_count_bar_sorted.png' 저장 완료.")
plt.show()

### 월별 활성 사용자 수 추이

In [ ]:
# 1. 'time' 컬럼(Unix 초)을 datetime 객체로 변환
df['event_time'] = pd.to_datetime(df['time'], unit='s')

# 2. 월(Month) 컬럼 생성 (MAU 집계용)
# to_period('M')은 '2025-06'과 같은 형태로 월을 추출합니다.
df['month'] = df['event_time'].dt.to_period('M')

# 3. 'user_type' 컬럼 생성 (로그인/비로그인 구분)
# PDF 설명에 따라, '$device:'로 시작하면 비로그인 사용자로 구분
df['user_type'] = df['distinct_id_event'].apply(
    lambda x: 'Non-logged-in' if str(x).startswith('$device:') else 'Logged-in'
)

print("  - 데이터 전처리 완료. 'event_time', 'month', 'user_type' 컬럼 생성됨.")

### 월별 활성 사용자 수(MAU) 추이

In [ ]:
# --- [제안 1] 월별 활성 사용자 수 (MAU) 추이 (막대 그래프) ---
# '월'을 기준으로 '고유한 사용자(distinct_id)'의 수를 집계합니다.

print("  - [시각화 1] MAU 추이 그래프 생성 중...")

# 1. 월별 고유 사용자(MAU) 집계
mau_data = df.groupby('month')['distinct_id_event'].nunique().reset_index()
mau_data.columns = ['Month', 'Unique Users']

# 2. plot을 위해 Period 타입을 문자열로 변경
mau_data['Month'] = mau_data['Month'].astype(str)

# 3. 시각화 (막대 그래프)
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=mau_data, 
    x='Month', 
    y='Unique Users', 
    palette='viridis' # 색상 팔레트
)

plt.title('월별 활성 사용자(MAU) 추이 (6월~10월)', fontsize=16)
plt.ylabel('고유 사용자 수 (명)', fontsize=12)
plt.xlabel('월', fontsize=12)

# 막대 위에 수치 표시
for p in ax.patches:
    ax.annotate(f'{int(p.get_height()):,}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 5), textcoords='offset points')

plt.show()

### 월별 사용자 구성(로그인 vs 비로그인)

In [ ]:
# --- [제안 2] 월별 사용자 구성 (로그인 vs 비로그인) (누적 막대 그래프) ---
# '월'과 'user_type' 두 가지 기준으로 고유 사용자 수를 집계합니다.

print("  - [시각화 2] 월별 사용자 구성(누적 막대) 그래프 생성 중...")

# 1. 월별, 사용자 유형별 고유 사용자 집계
monthly_composition = df.groupby(['month', 'user_type'])['distinct_id_event'].nunique().unstack().fillna(0)

# 2. 시각화 (누적 막대 그래프)
ax = monthly_composition.plot(
    kind='bar', 
    stacked=True, 
    figsize=(12, 7), 
    colormap='coolwarm' # 색상 팔레트
)

plt.title('월별 사용자 구성 (로그인/비로그인)', fontsize=16)
plt.ylabel('고유 사용자 수 (명)', fontsize=12)
plt.xlabel('월', fontsize=12)
plt.xticks(rotation=0) # x축 레이블 회전 방지
plt.legend(title='User Type')

# 누적 막대 그래프 위에 총 합계 표시
totals = monthly_composition.sum(axis=1)
for i, total in enumerate(totals):
    ax.text(i, total + 50, f'{int(total):,}', ha='center', weight='bold')

plt.show()

In [ ]:
# 1. 전체 사용자 수(Total) 계산
monthly_composition['Total'] = monthly_composition['Logged-in'] + monthly_composition['Non-logged-in']

# 2. 각 사용자 유형별 비율(%) 계산
monthly_composition['Logged-in (%)'] = (monthly_composition['Logged-in'] / monthly_composition['Total']) * 100
monthly_composition['Non-logged-in (%)'] = (monthly_composition['Non-logged-in'] / monthly_composition['Total']) * 100

# 3. 보기 좋게 출력 (소수점 둘째 자리까지 반올림)
print(" [!] 월별 사용자 구성 비율 요약표")
display(monthly_composition[['Logged-in', 'Non-logged-in', 'Total', 'Logged-in (%)', 'Non-logged-in (%)']].round(2))

### 사용자 활동 빈도 분포

In [ ]:
# --- [제안 3] 사용자 활동 빈도 분포 (히스토그램) ---

# 2. 데이터 준비 (이전 단계에서 만든 user_activity 사용)
# user_activity는 index가 사용자ID, value가 사용 횟수인 Series입니다.
user_activity = df.groupby('distinct_id_event').size()
clip_range = 100
user_activity_clipped = user_activity[user_activity < clip_range]

# 3. 그래프 그리기
fig, ax = plt.subplots(figsize=(10, 6))

# 핵심 변경 사항:
# - log_scale=True: X축을 로그 스케일로 변경
# - bins=30: 막대 개수를 줄여서 각 막대의 너비를 넓힘 (수치가 너무 크면 막대가 실처럼 얇아짐)
sns.histplot(
    data=user_activity_clipped, 
    bins=30,           # 막대 너비를 키우기 위해 bin 개수 조절
    kde=True,          # 밀도 추정 선 표시
    log_scale=True,    # X축 로그 스케일 적용
    ax=ax,
    color='steelblue', # 첫 번째 이미지와 유사한 색상
    alpha=0.6          # 투명도
)

# 4. 제목 및 라벨 설정
ax.set_title(f'사용자 활동 빈도 분포 (이벤트 {clip_range}회 미만)', fontsize=16)
ax.set_xlabel('사용자별 총 이벤트 수 (Log Scale)', fontsize=12)
ax.set_ylabel('사용자 수 (명)', fontsize=12)

# 5. (선택 사항) X축 눈금 포맷팅
# 첫 번째 이미지처럼 10^0, 10^1 형태로 보고 싶으면 아래 Formatter 코드를 주석 처리하세요.
# 만약 1, 10, 100 같은 숫자로 보고 싶다면 아래 주석을 해제하세요.
# ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
# ax.xaxis.set_minor_formatter(ticker.NullFormatter())

plt.show()

print(f"  - 시각화 완료: 총 {len(user_activity_clipped):,}명의 데이터가 표시되었습니다.")

In [ ]:
# 1. 구간(Bin) 설정
# 1회, 2~5회, 6~10회, 11~50회, 51~100회, 100회 초과 로 구간을 나눕니다.
bins = [0, 1, 5, 10, 50, 100, float('inf')]
labels = ['1회 (One-time)', '2~5회', '6~10회', '11~50회', '51~100회', '100회 초과']

# 2. 각 사용자의 활동 횟수(user_activity)를 구간별로 분류
# user_activity는 이전 단계에서 만든 Series (index: 사용자ID, value: 횟수)
df_dist = pd.cut(user_activity, bins=bins, labels=labels, right=True).value_counts().sort_index()

# 3. 데이터프레임으로 변환 및 비율 계산
activity_summary = pd.DataFrame({
    '사용자 수 (명)': df_dist,
    '비율 (%)': (df_dist / df_dist.sum()) * 100
})

# 4. 누적 비율(Cumulative %) 계산 - "하위 OO%가 여기에 속함"을 보여주기 위함
activity_summary['누적 비율 (%)'] = activity_summary['비율 (%)'].cumsum()

# 5. 결과 출력
print(" [!] 사용자 활동 빈도 구간별 요약표")
display(activity_summary.round(2))

# --- [PPT 발표용 핵심 문구 자동 생성] ---
total_users = len(user_activity)
one_time_users = activity_summary.loc['1회 (One-time)', '사용자 수 (명)']
one_time_pct = activity_summary.loc['1회 (One-time)', '비율 (%)']
under_5_pct = activity_summary.loc['2~5회', '누적 비율 (%)']

print("\n" + "="*40)
print("📢 [PPT 발표용 핵심 인사이트]")
print("="*40)
print(f"1. 전체 사용자의 과반수인 {one_time_pct:.1f}%({one_time_users:,}명)가 서비스를 단 '1회'만 사용했습니다.")
print(f"2. 범위를 5회 이하로 넓히면, 전체의 {under_5_pct:.1f}%가 '라이트 유저(Light User)'에 해당합니다.")
print(f"3. 즉, 대부분의 사용자는 호기심에 한두 번 써보고 이탈하거나 가끔씩만 방문하고 있습니다.")
print("="*40)

### 글로벌 시장 규모 성장 추이

In [ ]:
# 이미 상단에서 라이브러리들을 임포트했으므로, 데이터 정의와 시각화만 수행합니다.

# 1. 데이터 준비 (보고서 수치 기반)
data_market = {
    'Year': [2024, 2025, 2026, 2027, 2028, 2029, 2030, 2031, 2032],
    'Market Size (Billion $)': [1.75, 2.18, 2.73, 3.40, 4.25, 5.30, 6.61, 8.25, 10.30]
}
df_market = pd.DataFrame(data_market)

# 2. 색상 설정 (테마 설정 코드 제거함)
# 기본 회색, 시작(2024)과 끝(2032)은 강조색(보라색 계열)
colors = ['#B0B0B0'] * len(df_market) 
colors[0] = '#6A0DAD'  # 2024년
colors[-1] = '#6A0DAD' # 2032년

# 3. 그래프 그리기
plt.figure(figsize=(10, 6))

ax = sns.barplot(
    data=df_market,
    x='Year',
    y='Market Size (Billion $)',
    palette=colors
)

# 4. 텍스트 및 화살표 추가
plt.title('글로벌 AI 글쓰기 시장 규모 전망 (2024-2032)', fontsize=16, pad=20)
plt.ylabel('시장 규모 (10억 달러)', fontsize=12)
plt.xlabel('연도', fontsize=12)

# 막대 위 수치 표시
for p in ax.patches:
    ax.annotate(f'{p.get_height():.2f}', 
                (p.get_x() + p.get_width() / 2., p.get_height()), 
                ha='center', va='center', xytext=(0, 7), textcoords='offset points', fontsize=10)

# 성장 화살표 및 텍스트 추가 (2024 -> 2032)
# 좌표는 데이터 인덱스(0~8)와 y축 값 기준으로 조정
start_idx = 0       # 2024년의 x축 인덱스
end_idx = 8         # 2032년의 x축 인덱스
start_val = 1.75    # 2024년 값
end_val = 10.30     # 2032년 값

# 화살표 그리기
plt.annotate('', 
             xy=(end_idx, end_val + 1), 
             xytext=(start_idx, start_val + 1),
             arrowprops=dict(arrowstyle='->', lw=2, color='red', mutation_scale=20))

# 텍스트 추가 (화살표 중간 지점)
plt.text((start_idx + end_idx) / 2, (start_val + end_val) / 2 + 1.5, 
         '약 5.9배 성장\n(CAGR 24.8%)', 
         ha='center', color='red', fontsize=12, fontweight='bold')

plt.show()

### 지역별 시장 점유율 및 성장률

In [ ]:
# 1. 데이터 준비
data_region = {
    'Region': ['North America', 'Asia-Pacific', 'Europe', 'Latin America', 'Middle East & Africa'],
    'Market Share (%)': [37.5, 28.0, 22.0, 7.5, 5.0],
    'CAGR (%)': [22.0, 27.5, 20.5, 25.0, 24.0]
}
df_region = pd.DataFrame(data_region)

# 2. 그래프 그리기 (Subplots 활용)
# 여기서도 sns.set_theme()을 호출하지 않아 폰트 설정을 유지합니다.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# [왼쪽 그래프] 시장 점유율 (Market Share)
sns.barplot(ax=axes[0], data=df_region, x='Market Share (%)', y='Region', color='lightgray')
# 아시아-태평양만 강조 (파란색)
axes[0].patches[1].set_facecolor('steelblue') 
axes[0].set_title('지역별 시장 점유율 (2024)', fontsize=14)
axes[0].set_xlabel('점유율 (%)', fontsize=12)

# 수치 표시
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_width()}%', (p.get_width(), p.get_y() + p.get_height() / 2),
                     xytext=(5, 0), textcoords='offset points', va='center')


# [오른쪽 그래프] 연평균 성장률 (CAGR) - 여기가 핵심
sns.barplot(ax=axes[1], data=df_region, x='CAGR (%)', y='Region', color='lightgray')
# 아시아-태평양 강조 (빨간색 - 가장 빠른 성장)
axes[1].patches[1].set_facecolor('firebrick')
axes[1].set_title('지역별 연평균 성장률 전망 (CAGR)', fontsize=14)
axes[1].set_xlabel('성장률 (%)', fontsize=12)

# 수치 표시
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_width()}%', (p.get_width(), p.get_y() + p.get_height() / 2),
                     xytext=(5, 0), textcoords='offset points', va='center')

# 레이아웃 조정
plt.tight_layout()
plt.show()

### 사용자별 수락률 분석하기, 새로운 파일 만들기

In [ ]:
df['event_time'] = pd.to_datetime(df['time'], unit='s')

In [ ]:
# ---------------------------------------------------------
# [방법 1] 필터링 & 정렬 (가장 직관적인 형태)
# ---------------------------------------------------------

# 1. 우리가 관심 있는 이벤트 목록 정의
target_events = [
    'editor_run_paraphrasing', 
    'run_paraphrasing',
    'editor_selected_paraphrasing', 
    'selected_paraphrasing'
]

# 2. 해당 이벤트만 남기기 (Row Filtering)
filtered_df = df[df['event_name'].isin(target_events)].copy()

# 3. 보고 싶은 컬럼만 남기기 (Column Selection)
target_columns = [
    'distinct_id_event',                                # 사용자 ID
    'event_name',                                       # 이벤트명 (Run/Select)
    'llm_name',                                         # 모델명
    'maintenance',                                      # 교정 강도
    'field',                                            # 글 분야
    'total_paraphrasing_sentence_count',                # 동일 문장으로 재실행한 수(3의 배수로 측정: 3-1회 실행, 6-2회 실행)
    'response_time_seconds',                            # 소요 시간
    'event_time'                                        # timestamp  
]

final_df = filtered_df[target_columns]

# apply & lambda 활용하여 3으로 나누기
col_name = 'total_paraphrasing_sentence_count'
if col_name in final_df.columns:
    # 숫자인 경우 3으로 나누고, 비어있거나(NaN) 숫자가 아니면 그대로 둠(pass)
    final_df[col_name] = final_df[col_name].apply(
        lambda x: x / 3 if pd.notnull(x) and isinstance(x, (int, float)) else x
    )

# 4. 사용자별(distinct_id_event)로 모아서 보기 좋게 정렬
# 시간 순서(timestamp)가 있다면 같이 정렬해야 흐름이 보입니다.
sort_cols = ['distinct_id_event']
if 'event_time' in df.columns:
    sort_cols.append('event_time')

final_df = final_df.sort_values(by=sort_cols)

In [ ]:
final_df

In [ ]:
final_df.to_csv('./data/merge_acceptance_final.csv', index=False)